In [0]:
%python
df=spark.readStream.format("cloudFiles").option("cloudFiles.format","json").option("cloudFiles.schemaLocation","/Volumes/retailsales/bronze/autoloader/sales/schemaloc").option("rescuedDataColumn", "_rescued_data").option("schemaEvolutionMode","addNewColumns").option("badRecordsPath","/Volumes/retailsales/bronze/autoloader/sales/bad").load("/Volumes/retailsales/bronze/data/Sales/")

In [0]:
dbutils.fs.ls("/Volumes/retailsales/bronze/data/Sales/")

In [0]:
from pyspark.sql.functions import *

In [0]:
def process_batch(batch_df,batch_id):
    batch_df_monitoring=batch_df.select (col("_metadata.file_path").alias("file_path"),col("_metadata.file_name").alias("file_name"),"_rescued_data")
    batch_df_monitoring=batch_df_monitoring.groupBy("file_name","file_path").agg(count("*").alias("num_records"), sum(when( col("_rescued_data").isNotNull() , 1)).alias("resuced_records"))
    batch_df_monitoring.withColumns({"batch_id": lit(batch_id),
    "insert_date": current_timestamp(),"tbl_name":lit("sales")}).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("dq_table")

    batch_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("sales")

   


In [0]:
dic_df.write.format("delta").mode("append").saveAsTable("Schema_manager_bronze")

In [0]:
%sql
select * from dq_table 

In [0]:
fields=df.schema.fields
dic={}
list_nw=[]

for i,field in enumerate(fields):
    list_nw.append(field.name)
    list_nw.append(field.dataType.simpleString())
    list_nw.append("sales")
    dic[i]=list_nw
    list_nw=[]

print(dic.values())
dic_df=spark.createDataFrame(dic.values(),["col_name","type","table_nm"])

dic_df.createOrReplaceTempView("schema_temp")



    

In [0]:
test_df = (
    spark.read
    .schema(df.schema)
    .json("/Volumes/retailsales/bronze/data/Sales/new_schema_sales3.json")
)

display(test_df)

In [0]:
%sql
SELECT *
FROM retailsales.bronze.sales
ORDER BY created_timestamp DESC

In [0]:
%sql
if exists(select * from schema_temp)
begin
select * from schema_temp

except
select * from Schema_manager_bronze where table_nm='sales'
end



In [0]:
query =(df.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", "/Volumes/retailsales/bronze/autoloader/sales/checkpoint_new1") \
    .trigger(availableNow=True) \
    .start())

query.awaitTermination()

In [0]:
query.recentProgress

In [0]:
display(dbutils.fs.head(
    "dbfs:/Volumes/retailsales/bronze/data/Sales/new_schema_sales1.json"
))

In [0]:
display(dbutils.fs.ls(
    "/Volumes/retailsales/bronze/autoloader/sales/schemaloc"
))

In [0]:
import datetime
rescued_count=df.readtable("retailsales.bronze.sales").filter("_rescued_data").count()
load_date=datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, BooleanType, DateType, TimestampType

prod_schema=StructType([StructField('product_id', StringType(), True), StructField('sku', StringType(), True), StructField('product_name', StringType(), True), StructField('category', StringType(), True), StructField('subcategory', StringType(), True), StructField('brand', StringType(), True), StructField('cost_price', DecimalType(10,2), True), StructField('selling_price', DecimalType(10,2), True), StructField('tax_rate', DecimalType(5,2), True), StructField('weight_kg', DecimalType(6,2), True), StructField('is_active', BooleanType(), True), StructField('created_date', DateType(), True), StructField('last_updated', TimestampType(), True)])

In [0]:
def read_products():

    df=spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation","/Volumes/retailsales/bronze/autoloader/products/schemaloc").schema(prod_schema).load("/Volumes/retailsales/bronze/data/products/")

    df.writeStream.format("delta").option("checkpointLocation","/Volumes/retailsales/bronze/autoloader/products/checkpoint").trigger(availableNow=True).table("retailsales.bronze.products")



In [0]:
read_products()

In [0]:
cust_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("created_timestamp", TimestampType(), True)
])

In [0]:
def read_customers():

  
    df=spark.readStream.format("cloudFiles").option("cloudFiles.format","csv").option("cloudFiles.schemaLocation","/Volumes/retailsales/bronze/autoloader/customers/schemaloc").option("rescuedDataColumn", "_rescued_data").schema(cust_schema).option("schemaEvolutionMode","addNewColumns").option("badRecordsPath","/Volumes/retailsales/bronze/autoloader/customers/bad").load("/Volumes/retailsales/bronze/data/customers/")

    df.writeStream.format("delta").option("checkpointLocation","/Volumes/retailsales/bronze/autoloader/customers/checkpoint").option("mergeSchema","true").trigger(availableNow=True).table("retailsales.bronze.customers")


In [0]:
read_customers()

In [0]:
def read_stores():

  
    df=spark.readStream.format("cloudFiles").option("cloudFiles.format","json").option("cloudFiles.schemaLocation","/Volumes/retailsales/bronze/autoloader/stores/schemaloc").option("rescuedDataColumn", "_rescued_data").option("schemaEvolutionMode","addNewColumns").option("badRecordsPath","/Volumes/retailsales/bronze/autoloader/stores/bad").load("/Volumes/retailsales/bronze/data/stores/")

    df.writeStream.format("delta").option("checkpointLocation","/Volumes/retailsales/bronze/autoloader/stores/checkpoint").option("mergeSchema","true").trigger(availableNow=True).table("retailsales.bronze.stores")

In [0]:
read_stores()

In [0]:
%sql
select * from retailsales.bronze.customers

In [0]:
df=spark.readStream.table("retailsales.bronze.customers")
